# EEG Hyperscanning: from raw data to entropy and a Kuramoto model

This notebook builds the whole pipeline from scratch, with explanations at each step:

1. Load the raw BrainVision recording (one dyad, both participants multiplexed into one file)
2. Split into the two participants and keep only real scalp EEG channels
3. Attach a proper montage
4. Extract trial markers and cut the recording into per-trial epochs
5. Visualize the raw signal and the epochs
6. Compute permutation entropy per channel per trial
7. Build a Kuramoto oscillator model from first principles, to understand *why* Froese et al. (2024) predict that higher neural entropy should reduce inter-brain synchrony

Data: `ds007471` (OpenNeuro) — an EEG hyperscanning dataset of dyads performing a musical joint-action task. Only dyad `sub-01` is downloaded (see `data/ds007471/sub-01/`).

## 1. Load useful libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mne
from math import sin  # plain Python's math.sin is much faster than np.sin on single numbers


## 2. Load the raw recording

The file is in BrainVision format (`.eeg` binary data + `.vhdr` header + `.vmrk` markers). `mne.io.read_raw_brainvision` reads all three automatically from the `.vhdr` path.

In [ ]:
raw = mne.io.read_raw_brainvision(
    "data/ds007471/sub-01/eeg/sub-01_task-jointaction_eeg.vhdr",
    preload=True,
)

print(raw.info)
print(f"Sampling rate: {raw.info['sfreq']} Hz")
print(f"Duration: {raw.times[-1]:.1f} s")
print(f"Channels ({len(raw.ch_names)}): {raw.ch_names}")


## 3. Split into the two participants and drop the non-brain channels

This is a **hyperscanning** recording: both people in the dyad were recorded simultaneously into a single 64-channel file. Every channel name is suffixed `_R` or `_L` for the two people (32 channels each). Within each person's 32, most are real scalp EEG, but a few are not brain signal at all:

- **`VEOG_b`/`VEOG_a`** (vertical EOG) and **`HEOG_l`/`HEOG_r`** (horizontal EOG) sit near the eyes and record eye movements/blinks, not the brain. The eyeball acts like a small battery (the retina is more negatively charged than the cornea), so eye movements produce a much bigger electrical signal than the brain does — big enough to swamp nearby frontal EEG channels if left in. These channels exist specifically so eye-movement contamination can later be identified and cleaned out of the real EEG (e.g. with ICA) — they're not useless, just not brain data themselves.
- **`ref_lm`** is the linked-mastoid reference electrode, placed on the bone behind the ear where there's essentially no brain signal underneath. It's the electrical "zero point" every other channel is measured against, not a measurement of brain activity.

Feeding these into an entropy/inter-brain-synchrony calculation as if they were normal EEG would mean part of the result reflects blinking/eye movement rather than the brain, so they're excluded here, leaving 27 real scalp channels per participant.

In [ ]:
non_eeg = {"VEOG_b", "VEOG_a", "HEOG_l", "HEOG_r", "ref_lm"}

eeg_R = [ch for ch in raw.ch_names if ch.endswith("_R") and ch[:-2] not in non_eeg]
eeg_L = [ch for ch in raw.ch_names if ch.endswith("_L") and ch[:-2] not in non_eeg]

raw_R = raw.copy().pick(eeg_R)
raw_L = raw.copy().pick(eeg_L)

print(f"Participant R: {len(eeg_R)} EEG channels -> {eeg_R}")
print(f"Participant L: {len(eeg_L)} EEG channels -> {eeg_L}")


## 4. Rename channels to canonical 10-20 names and attach a montage

Downstream tools (e.g. HyPyP) expect each participant's data to look like ordinary single-subject EEG — plain electrode names (`F7`, not `F7_R`) with known scalp positions attached (a "montage"), since plotting functions project values onto a head model using those positions. Right now `raw_R`/`raw_L` still carry the `_R`/`_L` suffix, so we strip it and load the standard 10-20 positions for what's left.

In [ ]:
def to_canonical(raw_participant, suffix):
    raw_participant = raw_participant.copy()
    raw_participant.rename_channels(
        {ch: ch.removesuffix(suffix) for ch in raw_participant.ch_names}
    )
    raw_participant.set_montage("standard_1020", on_missing="warn")
    return raw_participant

raw_R = to_canonical(raw_R, "_R")
raw_L = to_canonical(raw_L, "_L")

print(raw_R.ch_names)
print(raw_L.ch_names)


## 5. Convert the recording's text markers into a numeric events array

The `.vmrk` file stores every marker as text — e.g. `Mk4=Stimulus,S105,214916,1,0` (type, description, sample number, duration, channel). Loading the file turns these into an `Annotations` object on `raw` (still text-based). `mne.Epochs` (used below) needs a different, numeric format instead, so we convert:

- `events`: array of `[sample_number, 0, event_code]` per marker (middle column is an unused legacy field for annotation-derived events).
- `event_id`: dict mapping each distinct marker description (prefixed with its type, e.g. `Stimulus/S105`) to the integer code used above.

`len(events)` counts *every* marker in the whole recording — not just the 40 trials, but every individual tone onset (`S128`/`S4` fire once per beat), block markers, trial-end markers, etc., so expect a large number. `event_id` is what we inspect to find out what marker types actually exist — cross-referenced against the dataset README, this is how the trial-onset code (`S105`, "the start of each trial, triggered by pressing the space bar") gets picked out for epoching below.

In [ ]:
events, event_id = mne.events_from_annotations(raw)

print(f"{len(events)} events")
print(f"Event types: {event_id}")


## 6. Cut the continuous recording into per-trial epochs

**What an epoch is:** the raw recording is one long continuous signal. An epoch is a fixed-length snippet cut out of that signal around a specific moment you care about (here, trial start), so that snippet 1 in trial A and snippet 1 in trial B line up in time relative to the same reference point — this is what lets you compare or average across trials.

Per the dataset README, `S105` marks the start of each trial (space bar press). We epoch a window around that marker for each participant separately — `raw_R`/`raw_L` share the same timeline/events since they're two channel-subsets of the same original recording, so the resulting epochs line up trial-for-trial between the two participants. `tmin`/`tmax` (0.5s before to 5s after trial start) are a rough starting window — narrow it once you know how long a 16-beat tone sequence actually runs.

In [ ]:
trial_onset_key = next(k for k in event_id if "105" in k)
tmin, tmax = -0.5, 5.0  # seconds relative to trial start; adjust once trial length is known

epochs_R = mne.Epochs(
    raw_R, events,
    event_id={trial_onset_key: event_id[trial_onset_key]},
    tmin=tmin, tmax=tmax, baseline=(tmin, 0),
    preload=True,
)
epochs_L = mne.Epochs(
    raw_L, events,
    event_id={trial_onset_key: event_id[trial_onset_key]},
    tmin=tmin, tmax=tmax, baseline=(tmin, 0),
    preload=True,
)

print(epochs_R)
print(f"{len(epochs_R)} trials x {len(epochs_R.ch_names)} channels x {epochs_R.get_data().shape[-1]} timepoints")


## 7. Visualize the continuous signal and the epochs

`raw.plot()` draws each channel as a horizontal trace stacked vertically, scrolling through time — the standard first look at EEG: scanning for obvious problems (flat/dead channels, slow drifts, huge spikes from movement) before trusting anything downstream. We point MNE's plotting at the `matplotlib` backend so the figure renders as a static image directly in the notebook, instead of opening a separate interactive window.

In [ ]:
mne.viz.set_browser_backend("matplotlib")

raw_R.plot(duration=10, n_channels=len(raw_R.ch_names), scalings="auto", title="Participant R - raw signal")


`epochs.plot()` looks like the same kind of trace view, but instead of one continuous scroll, the x-axis is chopped into the fixed-length windows we just cut out — each vertical dashed line is where one epoch ends and the next begins, so you can flip through trials one at a time and compare them directly.

In [ ]:
epochs_R.plot(n_epochs=5, n_channels=len(epochs_R.ch_names), scalings="auto", title="Participant R - epochs")


## 8. Compute permutation entropy per channel per trial

Permutation entropy (Bandt & Pompe, 2002) measures how *unpredictable the ordering* of a signal is. It slides a small window of `order` consecutive samples across the signal (here `order=3`, so triplets) and looks at each triplet's rank pattern — is it low→mid→high? high→mid→low? etc. There are `order! = 6` possible patterns for `order=3`. Across the whole window, you get a probability distribution over which pattern shows up; permutation entropy is the Shannon entropy of that distribution.

- A very regular signal (smooth oscillation, slow drift) keeps producing the *same* few rank patterns over and over → low entropy → predictable.
- A very irregular/noise-like signal produces all 6 patterns roughly equally often → high entropy → unpredictable.

With `normalize=True` the value is scaled to sit between 0 (fully predictable) and 1 (maximally unpredictable — every pattern equally likely), which makes values comparable across channels/trials regardless of amplitude. In the entropic-brain framing this project builds on (Carhart-Harris et al. 2014; Froese et al. 2024), a higher value here means "more disordered/complex" neural activity in that window — the quantity the hypothesis says should trade off against inter-brain synchrony.

In [ ]:
import antropy as ant

def permutation_entropy_per_channel(epochs, order=3):
    data = epochs.get_data()  # shape: (n_epochs, n_channels, n_times)
    return np.array([
        ant.perm_entropy(data[:, ch, :], order=order, normalize=True)
        for ch in range(data.shape[1])
    ]).T  # shape: (n_epochs, n_channels)

pe_R = permutation_entropy_per_channel(epochs_R)
pe_L = permutation_entropy_per_channel(epochs_L)

for label, pe, ep in [("R", pe_R, epochs_R), ("L", pe_L, epochs_L)]:
    print(f"Participant {label} - permutation entropy (0=predictable, 1=unpredictable)")
    print(f"  overall mean: {pe.mean():.3f}")
    print(f"  per trial (avg across channels): {np.round(pe.mean(axis=1), 3)}")
    print(f"  per channel (avg across trials):")
    for ch, val in zip(ep.ch_names, pe.mean(axis=0)):
        print(f"    {ch}: {val:.3f}")
    print()


## 9. Kuramoto model, built from scratch

This section is a simulation, independent of the real EEG data above — the goal is to build hands-on intuition for the mechanism Froese, Loh & Putri (2024) propose: *why* would higher individual neural entropy cause lower inter-brain synchrony?

We build it up in stages, each one small enough to fully understand before adding the next layer of complexity:

1. One oscillator on its own
2. Two oscillators, coupled
3. A population of oscillators (one "brain")
4. Two populations, coupled to each other (two "brains")
5. A way to measure synchrony between the two brains
6. Put it together: does more disorder (entropy) mean less synchrony?

Everything here uses plain Python loops rather than vectorized numpy tricks — slower to run, but every line maps directly onto the math.

### 9.1 One oscillator

A "phase oscillator" is the simplest possible model of something that rhythmically repeats — think of a clock hand, or one neuron firing in a rhythm. Its state is a single angle `θ` (theta) that just keeps increasing over time, at some speed `ω` (omega), its **natural frequency**:

```
dθ/dt = ω
```

That's it — no interaction yet. We integrate this numerically with the simplest possible method, **Euler integration**: repeatedly nudge `θ` forward by `(dθ/dt) * dt` for a small timestep `dt`. To turn the angle into something that looks like a signal (like an EEG trace), we take `sin(θ)` — as `θ` sweeps around the circle, `sin(θ)` traces out a wave.

In [ ]:
omega = 2 * np.pi * 40   # natural frequency: 40 Hz, converted to radians/second
dt = 0.001                # timestep, in seconds
n_steps = 200              # simulate 0.2 seconds total

theta = 0.0
theta_history = [theta]

for _ in range(n_steps):
    dtheta_dt = omega                 # the differential equation: dtheta/dt = omega
    theta = theta + dtheta_dt * dt    # Euler step: theta(t+dt) = theta(t) + dtheta/dt * dt
    theta_history.append(theta)

theta_history = np.array(theta_history)
times = np.arange(n_steps + 1) * dt

plt.figure(figsize=(7, 3))
plt.plot(times, np.sin(theta_history))
plt.xlabel("time (s)")
plt.ylabel("sin(theta)")
plt.title("A single 40 Hz oscillator")
plt.show()


### 9.2 Two oscillators, coupled

Now give each oscillator a slightly different natural frequency, and let them influence each other. The **Kuramoto coupling term** is what turns two independent oscillators into an interacting pair:

```
dθ₁/dt = ω₁ + K · sin(θ₂ − θ₁)
dθ₂/dt = ω₂ + K · sin(θ₁ − θ₂)
```

Focus on `sin(θ₂ − θ₁)`: if oscillator 2 is "ahead" of oscillator 1 (their phase difference is a small positive angle), `sin` of that is positive, which speeds oscillator 1 up to catch up. If oscillator 2 is "behind," it's negative, slowing oscillator 1 down. So this term always pulls oscillator 1 toward oscillator 2's phase — and symmetrically, the second equation pulls oscillator 2 toward oscillator 1. `K` is the coupling strength: how hard they pull on each other. If `K` is large enough relative to how different `ω₁` and `ω₂` are, the pull wins and they lock into the same rhythm — this is called **phase locking** or **synchronization**. If `K` is too weak, their frequency difference wins and they just drift apart.

In [ ]:
def simulate_two_oscillators(omega1, omega2, K, dt=0.001, n_steps=500):
    theta1, theta2 = 0.0, np.pi  # start out of phase, to make locking visible
    diffs = []
    for _ in range(n_steps):
        dtheta1_dt = omega1 + K * np.sin(theta2 - theta1)
        dtheta2_dt = omega2 + K * np.sin(theta1 - theta2)
        theta1 = theta1 + dtheta1_dt * dt
        theta2 = theta2 + dtheta2_dt * dt
        # wrap the difference into [-pi, pi] so it's easy to read on a plot
        diff = (theta1 - theta2 + np.pi) % (2 * np.pi) - np.pi
        diffs.append(diff)
    return np.array(diffs)

omega1 = 2 * np.pi * 39   # 39 Hz
omega2 = 2 * np.pi * 41   # 41 Hz
times = np.arange(500) * 0.001

plt.figure(figsize=(7, 3))
plt.plot(times, simulate_two_oscillators(omega1, omega2, K=2.0), label="K = 2.0 (too weak)")
plt.plot(times, simulate_two_oscillators(omega1, omega2, K=20.0), label="K = 20.0 (strong enough)")
plt.axhline(0, color="gray", linewidth=0.5)
plt.xlabel("time (s)")
plt.ylabel("phase difference theta1 - theta2 (rad)")
plt.title("Phase locking depends on coupling strength vs. frequency mismatch")
plt.legend()
plt.show()


### 9.3 A population of oscillators (one "brain")

Generalize from 2 oscillators to `N`, all pulling on each other (all-to-all coupling). Oscillator `i`'s equation now sums the pull from every other oscillator `j`:

```
dθᵢ/dt = ωᵢ + (K / (N-1)) · Σⱼ sin(θⱼ − θᵢ)
```

We divide by `N-1` so that the total pull on one oscillator doesn't just keep growing as you add more oscillators to the population — each individual pull gets weaker as the group gets bigger, but the *average* pull stays comparable to the two-oscillator case.

To measure "how synchronized is this whole population," the standard tool is the **Kuramoto order parameter** `r`: treat each oscillator's phase as a point on the unit circle (`e^(iθ)`), average all `N` points together, and take the length of the resulting vector.

- If all oscillators have the *same* phase, their points all stack on top of each other, and the average has length 1 → **r = 1** (fully synchronized).
- If phases are spread evenly around the circle, they cancel out, and the average has length ~0 → **r ≈ 0** (incoherent).

In [ ]:
def order_parameter(theta):
    """theta: array of N phases (radians). Returns r, between 0 (incoherent) and 1 (locked)."""
    return np.abs(np.mean(np.exp(1j * theta)))

def simulate_population(omegas, K, dt=0.001, n_steps=1000, rng=None):
    rng = rng or np.random.default_rng()
    n = len(omegas)
    theta = rng.uniform(0, 2 * np.pi, size=n)  # random starting phases
    r_history = [order_parameter(theta)]

    for _ in range(n_steps):
        new_theta = np.empty(n)
        for i in range(n):
            coupling = sum(np.sin(theta[j] - theta[i]) for j in range(n) if j != i)
            new_theta[i] = theta[i] + dt * (omegas[i] + (K / (n - 1)) * coupling)
        theta = new_theta
        r_history.append(order_parameter(theta))

    return np.array(r_history)

rng = np.random.default_rng(0)
N = 10
omegas = 2 * np.pi * rng.uniform(39, 41, size=N)  # narrow spread, like the paper's C1
r_history = simulate_population(omegas, K=15.0, rng=rng)  # strong coupling, so locking is clearly visible

times = np.arange(len(r_history)) * 0.001
plt.figure(figsize=(7, 3))
plt.plot(times, r_history)
plt.xlabel("time (s)")
plt.ylabel("order parameter r")
plt.title(f"{N} coupled oscillators (narrow frequency spread) synchronizing over time")
plt.ylim(0, 1)
plt.show()


### 9.4 Two populations, coupled to each other (two "brains")

Now the hyperscanning part: two separate groups of oscillators, each strongly coupled *within* itself (`K_intra`), and more weakly coupled *to the other group* (`K_inter`) — representing however the two people influence each other (shared stimuli, mutual observation, in Froese et al.'s framing). Oscillator `i` in brain 1 now gets pulled by two sums: its own brain's oscillators, and all of brain 2's oscillators.

We track each brain's own order parameter over time (`r₁`, `r₂`) — this tells us how synchronized *each individual brain* is internally (this is the "intra-brain" side of things).

In [ ]:
def simulate_two_brains(omegas1, omegas2, K_intra, K_inter, dt=0.001, n_steps=2000, burn_in=1000, rng=None):
    rng = rng or np.random.default_rng()
    n = len(omegas1)
    theta1 = rng.uniform(0, 2 * np.pi, size=n)
    theta2 = rng.uniform(0, 2 * np.pi, size=n)

    r1_history, r2_history = [], []
    phase_history1 = np.empty((n, n_steps - burn_in))
    phase_history2 = np.empty((n, n_steps - burn_in))

    for step in range(n_steps):
        new_theta1 = np.empty(n)
        new_theta2 = np.empty(n)
        for i in range(n):
            intra = sum(sin(theta1[j] - theta1[i]) for j in range(n) if j != i)
            inter = sum(sin(theta2[j] - theta1[i]) for j in range(n))
            new_theta1[i] = theta1[i] + dt * (omegas1[i] + K_intra / (n - 1) * intra + K_inter / n * inter)
        for i in range(n):
            intra = sum(sin(theta2[j] - theta2[i]) for j in range(n) if j != i)
            inter = sum(sin(theta1[j] - theta2[i]) for j in range(n))
            new_theta2[i] = theta2[i] + dt * (omegas2[i] + K_intra / (n - 1) * intra + K_inter / n * inter)
        theta1, theta2 = new_theta1, new_theta2

        r1_history.append(order_parameter(theta1))
        r2_history.append(order_parameter(theta2))
        if step >= burn_in:
            phase_history1[:, step - burn_in] = theta1
            phase_history2[:, step - burn_in] = theta2

    return phase_history1, phase_history2, np.array(r1_history), np.array(r2_history)

rng = np.random.default_rng(1)
omegas1 = 2 * np.pi * rng.uniform(39, 41, size=N)
omegas2 = 2 * np.pi * rng.uniform(39, 41, size=N)

phase_history1, phase_history2, r1_history, r2_history = simulate_two_brains(
    omegas1, omegas2, K_intra=15.0, K_inter=5.0, n_steps=1000, burn_in=0, rng=rng,
)

times = np.arange(len(r1_history)) * 0.001
plt.figure(figsize=(7, 3))
plt.plot(times, r1_history, label="brain 1 (r1)")
plt.plot(times, r2_history, label="brain 2 (r2)")
plt.xlabel("time (s)")
plt.ylabel("order parameter r")
plt.title("Two coupled brains, each synchronizing internally")
plt.ylim(0, 1)
plt.legend()
plt.show()


### 9.5 Measuring synchrony *between* the two brains

The order parameter `r` only tells you how coherent oscillators are *within one group* — it can't tell you whether brain 1 and brain 2 are moving together, since they're two separate populations with their own `r`. For that, we need to directly compare two phase time series and ask "do they covary?" — the same question HyPyP's inter-brain connectivity answers for real EEG channels.

The tool for this is **circular correlation** (Jammalamadaka & SenGupta) — like Pearson correlation, but built for angles, where 359° and 1° are almost the same value rather than far apart:

1. Find each signal's circular mean (its average angle, computed correctly by averaging points on the unit circle, not the raw numbers).
2. Look at how far each timepoint deviates from that mean (`sin(θ − mean)` handles the wraparound correctly).
3. Correlate those deviations, the same way you'd correlate two ordinary variables.

We apply this to every pair of oscillators — within brain 1, within brain 2, and across the two brains — and average each group separately to get one **intra-brain** and one **inter-brain** number.

In [ ]:
def circular_correlation(theta_a, theta_b):
    """Jammalamadaka & SenGupta circular correlation between two phase time series."""
    mean_a = np.angle(np.mean(np.exp(1j * theta_a)))
    mean_b = np.angle(np.mean(np.exp(1j * theta_b)))
    deviation_a = np.sin(theta_a - mean_a)
    deviation_b = np.sin(theta_b - mean_b)
    numerator = np.sum(deviation_a * deviation_b)
    denominator = np.sqrt(np.sum(deviation_a ** 2) * np.sum(deviation_b ** 2))
    return numerator / denominator

# every within-brain pair (excluding an oscillator with itself)
intra_1 = [circular_correlation(phase_history1[i], phase_history1[j])
           for i in range(N) for j in range(N) if i != j]
intra_2 = [circular_correlation(phase_history2[i], phase_history2[j])
           for i in range(N) for j in range(N) if i != j]

# every cross-brain pair
inter = [circular_correlation(phase_history1[i], phase_history2[j])
          for i in range(N) for j in range(N)]

print(f"Intra-brain CCorr (avg of both brains): {np.mean(intra_1 + intra_2):.3f}")
print(f"Inter-brain CCorr: {np.mean(inter):.3f}")


### 9.6 Put it together: narrow vs. wide frequency spread

Now we run the actual comparison Froese et al. (2024) ran in their proof-of-concept: two conditions that differ only in how *widely spread* the oscillators' natural frequencies are.

- **C1** (narrow spread, [39, 41] Hz): the oscillators mostly agree on a frequency already, so coupling has an easy job — this stands in for **low neural entropy**.
- **C2** (wide spread, [30, 50] Hz): the oscillators disagree much more about what frequency to settle on, so coupling has to fight harder to pull them together — this stands in for **high neural entropy** (a broader range of available neural states, in the paper's own framing).

We use the paper's actual coupling values (`K_intra=1.0`, `K_inter=0.5`, weaker than the strong values used above just to make the demo plots readable) and repeat each condition several times with fresh random frequencies and starting phases, to average out noise.

**A note on runtime and exact numbers:** this pure-Python version is much slower than a vectorized one, so the repeat count below is modest — enough to see the pattern, not a polished estimate. The paper reports `C1: 0.32, C2: 0.045` for inter-brain CCorr specifically, but doesn't fully specify integration duration/timestep or exact coupling normalization — details that swing the *absolute* CCorr values a lot in testing. Don't expect an exact match. What should hold up reliably is the *direction*: C2 (wider frequency spread, "higher entropy") shows lower synchrony than C1, both within brains and between them.

In [ ]:
def run_condition(f_min, f_max, n_sims, K_intra=1.0, K_inter=0.5, rng=None):
    rng = rng or np.random.default_rng()
    intra_scores, inter_scores = [], []

    for _ in range(n_sims):
        omegas1 = 2 * np.pi * rng.uniform(f_min, f_max, size=N)
        omegas2 = 2 * np.pi * rng.uniform(f_min, f_max, size=N)

        ph1, ph2, _, _ = simulate_two_brains(
            omegas1, omegas2, K_intra=K_intra, K_inter=K_inter,
            n_steps=1500, burn_in=750, rng=rng,
        )

        intra_1 = [circular_correlation(ph1[i], ph1[j]) for i in range(N) for j in range(N) if i != j]
        intra_2 = [circular_correlation(ph2[i], ph2[j]) for i in range(N) for j in range(N) if i != j]
        inter_pairs = [circular_correlation(ph1[i], ph2[j]) for i in range(N) for j in range(N)]

        intra_scores.append(np.mean(intra_1 + intra_2))
        inter_scores.append(np.mean(inter_pairs))

    return np.mean(intra_scores), np.mean(inter_scores)

rng = np.random.default_rng(42)
N_SIMS = 20  # modest, for pure-Python speed -- increase if you're willing to wait longer

conditions = {
    "C1 (narrow freq, low entropy)": (39.0, 41.0),
    "C2 (wide freq, high entropy)": (30.0, 50.0),
}

results = {}
for label, (f_min, f_max) in conditions.items():
    intra, inter = run_condition(f_min, f_max, N_SIMS, rng=rng)
    results[label] = {"intra": intra, "inter": inter}
    print(f"{label}: intra-brain CCorr = {intra:.3f}, inter-brain CCorr = {inter:.3f}")


In [ ]:
labels = list(results.keys())
intra_vals = [results[l]["intra"] for l in labels]
inter_vals = [results[l]["inter"] for l in labels]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width / 2, intra_vals, width, label="intra-brain CCorr")
ax.bar(x + width / 2, inter_vals, width, label="inter-brain CCorr")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("CCorr")
ax.set_title("Wider natural-frequency range (higher entropy) -> lower synchrony")
ax.legend()
plt.show()


Their full code is at [gitlab.com/oist-ecsu/ibdesync](https://gitlab.com/oist-ecsu/ibdesync) if you want to compare against their exact implementation.